# 13.4 - Loops & Cycles

**Phase:** 13 - LangGraph / Stateful Workflows

**Status:** VERIFIED

---

## 1. What Are We Solving?

Loops let a graph revisit nodes until a condition is met — retries, iterative refinement, and convergence. The graph's back-edge replaces a Python `while` loop and keeps state checkpointable.

## 2. Why Does This Matter?

## 3. Prerequisites

Units 13.1-13.3.

## 4. Learning Objectives

By the end of this unit, you should be able to:
- Build a refinement loop with a back-edge
- Always guard loops with a max-iteration counter
- Add convergence metrics to state
- Build a self-correcting QA pipeline using Groq

## 5. Mental Model

A loop is a feedback cycle: do work -> check the result -> not good enough? -> go back and redo -> repeat until satisfied (or until the guard trips).


## 6. Setup

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

from typing import TypedDict, Annotated
import operator
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

GROQ_MODEL = os.environ.get("GROQ_MODEL", "openai/gpt-oss-20b")


def llm(prompt: str, model: str = GROQ_MODEL, temperature: float = 0.0) -> str:
    """One-shot Groq call with a deterministic mock fallback."""
    if not os.environ.get("GROQ_API_KEY"):
        return "mock: LangGraph is a stateful orchestration framework."
    try:
        from langchain_groq import ChatGroq
        chat = ChatGroq(model=model, temperature=temperature)
        return chat.invoke(prompt).content.strip()
    except Exception as e:  # network / quota / model errors -> never crash the notebook
        return f"[llm-error: {type(e).__name__}]"

print("Groq model:", GROQ_MODEL)
print("GROQ_API_KEY present:", bool(os.environ.get("GROQ_API_KEY")))


Groq model: openai/gpt-oss-20b
GROQ_API_KEY present: True


## 7. Refinement Loop with Convergence Guard

Draft quality rises each iteration; the router goes back to `generate` until quality crosses the threshold or steps run out.

In [2]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

trace_quality = []


class RefineState(TypedDict):
    query: str
    draft: str | None
    quality: float
    max_steps: int
    step: int


def generate_draft(state):
    state["step"] += 1
    if state["draft"] is None:
        state["draft"] = f"Initial draft about: {state['query']}"
    else:
        head = " ".join(state["draft"].split()[:8])
        state["draft"] = f"Draft v{state['step']}: {head} ..."
    state["quality"] = min(0.3 + 0.25 * state["step"], 1.0)
    trace_quality.append(state["quality"])
    return state


def should_continue(state):
    if state["quality"] >= 0.8 or state["step"] >= state["max_steps"]:
        return "done"
    return "refine"


def finalize(state):
    state["draft"] = f"FINAL: {state['draft']} (quality={state['quality']:.2f})"
    return state


g = StateGraph(RefineState)
g.add_node("generate", generate_draft)
g.add_node("finalize", finalize)
g.add_edge(START, "generate")
g.add_conditional_edges("generate", should_continue,
                         {"refine": "generate", "done": "finalize"})
g.add_edge("finalize", END)

app = g.compile()
res = app.invoke({"query": "Explain state machines", "draft": None,
                  "quality": 0.0, "max_steps": 5, "step": 0})
print("iterations:", res["step"])
print(res["draft"])

plt.figure(figsize=(5, 2.5))
plt.plot(range(1, len(trace_quality) + 1), trace_quality, marker="o")
plt.axhline(0.8, color="r", linestyle="--", label="threshold")
plt.xlabel("iteration"); plt.ylabel("quality"); plt.title("Convergence"); plt.legend()
plt.grid(alpha=0.3)
plt.show()


iterations: 2
FINAL: Draft v2: Initial draft about: Explain state machines ... (quality=0.80)


C:\Users\PC\AppData\Local\Temp\ipykernel_12372\3380678402.py:58: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Self-Correcting QA with Groq

Retrieve -> answer -> evaluate -> refine query and loop until the score clears the bar or we hit the guard.

In [3]:
CORPUS = {
    "langgraph": "LangGraph is a graph-based orchestration framework for stateful LLM agents, with checkpointing and human-in-the-loop support.",
    "rag": "RAG (retrieval-augmented generation) grounds LLM answers by retrieving relevant documents before generating.",
    "gradient": "Gradient descent iteratively updates parameters to minimize a loss function using its gradients.",
}


class QALoopState(TypedDict):
    question: str
    refined_question: str
    context: str
    answer: str | None
    score: float
    iterations: int
    max_iterations: int
    history: list


def retrieve(state):
    q = state["refined_question"].lower()
    best = ""; best_score = 0
    for text in CORPUS.values():
        s = sum(1 for w in q.split() if w in text.lower())
        if s > best_score:
            best, best_score = text, s
    state["context"] = best if best_score else "No relevant document found."
    return state


def generate(state):
    ans = llm(f"Context: {state['context']}\nQuestion: {state['refined_question']}\nAnswer concisely.")
    return {"answer": ans,
            "history": state["history"] + [(state["iterations"], state["score"], ans)]}


def evaluate(state):
    prompt = (f"Rate from 0.0 to 1.0 how well this answer addresses the question. Reply with a number only.\n"
              f"Q: {state['refined_question']}\nA: {state['answer']}")
    raw = llm(prompt).strip().replace(',', '.')
    try:
        state["score"] = max(0.0, min(1.0, float(raw)))
    except ValueError:
        state["score"] = 0.5
    return state


def refine(state):
    state["iterations"] += 1
    state["refined_question"] = "More precisely, " + state["refined_question"]
    return state


def done(state):
    return state


def after_eval(state):
    if state["score"] >= 0.7 or state["iterations"] >= state["max_iterations"]:
        return "done"
    return "refine"


g = StateGraph(QALoopState)
for n, fn in (("retrieve", retrieve), ("generate", generate), ("evaluate", evaluate),
              ("refine", refine), ("done", done)):
    g.add_node(n, fn)
g.add_edge(START, "retrieve")
g.add_edge("retrieve", "generate")
g.add_edge("generate", "evaluate")
g.add_conditional_edges("evaluate", after_eval, {"done": "done", "refine": "refine"})
g.add_edge("refine", "retrieve")
g.add_edge("done", END)
app = g.compile()

out = app.invoke({"question": "What is RAG?", "refined_question": "What is RAG?", "context": "",
                  "answer": None, "score": 0.0, "iterations": 0, "max_iterations": 2, "history": []})
print("iterations:", out["iterations"], "| final score:", round(out["score"], 2))
print("FINAL ANSWER:", out["answer"])
print("\niteration history:")
for it in out["history"]:
    print(f"  iter {it[0]} (score {it[1]:.2f}): {it[2][:60]}...")


iterations: 0 | final score: 0.95
FINAL ANSWER: RAG = **Retrieval‑Augmented Generation** – a method where an LLM first fetches relevant external documents or data (retrieval) and then uses that information to produce a more accurate, context‑aware response (generation).

iteration history:
  iter 0 (score 0.00): RAG = **Retrieval‑Augmented Generation** – a method where an...



## Common Mistakes

- **Return `None` from a node** — the graph silently drops the update. Always `return state`.
- **Mutating state in the router** — routers must be side-effect free.
- **Forgetting the terminal condition** — cycles run forever without an iteration guard.
- **Typo in a state key** — `TypedDict` catches it at compile time; plain dicts do not.

## Debugging

| Symptom | Likely Cause | Fix |
|---|---|---|
| `KeyError` on state | Field name mismatch | Match state keys to the `TypedDict` exactly |
| Graph won't compile | Node/referenced name typo | Check every string passed to `add_node`/`add_edge` |
| Node output ignored | Node returns `None` or a partial dict | Always `return state` (or a merge-able partial) |
| Infinite loop | No convergence guard | Add `max_steps` to state and check it in the router |
| Wrong branch taken | Router priority bug | Unit-test the router on every input variant |

## Best Practices

- Define all state fields upfront with defaults in a `TypedDict`.
- Keep node functions pure and focused: one responsibility each.
- Name nodes descriptively (`retrieve`, `generate`, not `step1`).
- Always add an iteration guard on loops.
- Inspect the graph with `app.get_graph().draw_mermaid()`.

## Hands-On Practice

1. **Basic:** Rerun the examples with new inputs; verify the trace.
2. **Guided:** Add a node that validates output before terminating.
3. **Independent:** Build a 3-step pipeline (fetch -> process -> summarize) with a retry node.
4. **Realistic:** Turn the example into a multi-department support agent.
5. **Challenge:** Save/load the state dict to JSON and resume the workflow from a checkpoint.

## Exit Criteria

- You can explain and build the concept from scratch.
- You can debug the associated failure modes.
- You know when to reach for this tool vs. a plain function.
